In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [6]:
np.random.seed(42)
n_samples = 2000

sqft = np.random.normal(1800, 700, n_samples).clip(400, 6000)
bedrooms = np.random.randint(1, 6, n_samples)
bathrooms = np.random.randint(1, 4, n_samples) + np.random.choice([0, 0.5], n_samples)
age = np.random.uniform(0, 80, n_samples)
distance_to_city = np.random.uniform(0.5, 40, n_samples)
crime_rate = np.random.exponential(3, n_samples).clip(0, 25)
school_rating = np.random.uniform(1, 10, n_samples)


In [7]:
price = (
    50000
    + 180 * sqft
    + 8000 * bedrooms
    + 12000 * bathrooms
    - 800 * age
    - 1500 * distance_to_city
    - 2000 * crime_rate
    + 9000 * school_rating
    + np.random.normal(0, 25000, n_samples)
).clip(50000, None)

df = pd.DataFrame({
    "sqft": sqft,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "age_years": age,
    "distance_to_city_miles": distance_to_city,
    "crime_rate": crime_rate,
    "school_rating": school_rating,
    "price": price
})
print("Dataset shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:\n", df.head())
print("\nSummary statistics:\n", df.describe())

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=120)
plt.close()

plt.figure(figsize=(7, 5))
plt.scatter(df["sqft"], df["price"], alpha=0.2, s=10)
plt.xlabel("Square Footage")
plt.ylabel("House Price (USD)")
plt.title("Square Footage vs House Price")
plt.tight_layout()
plt.savefig("income_vs_price.png", dpi=120)
plt.close()

X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("\nModel Intercept:", round(model.intercept_, 3))
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)
print("\nFeature Coefficients (standardized):\n", coef_df)

y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nMSE:  {mse:,.2f}")
print(f"RMSE: ${rmse:,.0f}  (typical prediction error in dollars)")
print(f"MAE:  ${mae:,.0f}")
print(f"R2:   {r2:.4f}")

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=12)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label="Perfect prediction")
plt.xlabel("Actual House Price (USD)")
plt.ylabel("Predicted House Price (USD)")
plt.title("Actual vs Predicted House Values")
plt.legend()
plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=120)
plt.close()

residuals = y_test - y_pred
plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.3, s=12)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel("Predicted House Price (USD)")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.tight_layout()
plt.savefig("residual_plot.png", dpi=120)
plt.close()

print("\nAll plots saved.")

Dataset shape: (2000, 8)

Columns: ['sqft', 'bedrooms', 'bathrooms', 'age_years', 'distance_to_city_miles', 'crime_rate', 'school_rating', 'price']

First 5 rows:
           sqft  bedrooms  bathrooms  age_years  distance_to_city_miles  \
0  2147.699907         3        3.5  57.175687                7.658113   
1  1703.214989         3        3.5  48.303421               32.506980   
2  2253.381977         3        3.0  23.914092                5.834006   
3  2866.120899         4        3.0  32.865312               19.011740   
4  1636.092638         5        2.5  17.415984               37.573840   

   crime_rate  school_rating          price  
0    1.165145       8.136121  536713.193151  
1    0.848991       8.239568  398140.726142  
2    0.027139       2.863763  493149.072928  
3    0.805488       4.023225  631480.492708  
4    0.444555       2.208146  367806.300054  

Summary statistics:
               sqft     bedrooms    bathrooms    age_years  \
count  2000.000000  2000.000000 

In [8]:
print("\nEnter details for the house you want to price:")
new_house = {
    "sqft": float(input("Square foot: ")),
    "bedrooms": int(input("Bedrooms: ")),
    "bathrooms": float(input("Bathrooms: ")),
    "age_years": float(input("Age (years): ")),
    "distance_to_city_miles": float(input("Distance to city (miles): ")),
    "crime_rate": float(input("Crime rate (0-10 scale): ")),
    "school_rating": float(input("School rating (1-10): "))
}

new_house_df = pd.DataFrame({k: [v] for k, v in new_house.items()})

new_house_df = new_house_df[X.columns]
new_house_scaled = scaler.transform(new_house_df)
predicted_price = model.predict(new_house_scaled)[0]

print("\n--- New House Prediction ---")
for feature, value in new_house.items():
    print(f"{feature}: {value}")
print(f"Predicted Price: ${predicted_price:,.0f}")


Enter details for the house you want to price:
Square foot: 30000
Bedrooms: 65
Bathrooms: 50
Age (years): 3
Distance to city (miles): 2
Crime rate (0-10 scale): 2
School rating (1-10): 8

--- New House Prediction ---
sqft: 30000.0
bedrooms: 65
bathrooms: 50.0
age_years: 3.0
distance_to_city_miles: 2.0
crime_rate: 2.0
school_rating: 8.0
Predicted Price: $6,630,472


In [10]:
X_train

,sqft,bedrooms,bathrooms,age_years,distance_to_city_miles,crime_rate,school_rating
81,2049.978800,3,2.5,44.968871,16.030232,2.982815,9.705873
915,1995.315068,2,1.5,29.088916,7.101805,5.489631,3.398951
1018,1575.756884,2,1.5,26.155368,24.555465,0.524467,8.167112
380,1212.194710,1,2.0,76.638348,14.334346,1.379545,6.986837
1029,1084.045204,5,3.5,62.958064,21.412712,3.706637,3.316078
...,...,...,...,...,...,...,...
1130,2024.950051,3,2.0,35.042693,8.023051,0.451221,1.302033
1294,1857.280551,4,1.5,1.040817,37.546536,3.907012,3.529251
860,1942.046115,4,3.0,75.251200,34.323263,5.956194,5.515327
1459,2271.226946,4,2.5,74.553019,20.701738,2.389900,5.791588
